# الدرس 10 - وكلاء الذكاء الاصطناعي في الإنتاج

في هذا الدرس ستتعلم **أنماط الإنتاج** لوكلاء الذكاء الاصطناعي باستخدام Microsoft Agent Framework (Python). نغطي:

- **قابلية المراقبة** — إضافة معلومات زمنية وتسجيل للتفاعلات مع الوكيل
- **التقييم** — استخدام وكيل للتقييم لتقييم جودة الاستجابة
- **إدارة التكلفة** — استراتيجيات لتحسين عدد الرموز واختيار النموذج

السيناريو هو **وكيل سفر** يساعد المستخدمين على تخطيط الرحلات، مع إضافة المراقبة والتقييم كطبقات فوق ذلك.


## الإعداد


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## اعتبارات الإنتاج

يتطلب نقل وكلاء الذكاء الاصطناعي من نماذج أولية إلى بيئة الإنتاج اهتمامًا دقيقًا بثلاثة ركائز:

1. **القابلية للملاحظة** — تحتاج إلى رؤية ما يفعله الوكيل، والمدة التي يستغرقها، والأدوات التي يستدعيها. بدون التتبع والتسجيل، يصبح تصحيح مشكلات الإنتاج شبه مستحيل.

2. **التقييم** — تضمن فحوصات الجودة الآلية بقاء استجابات الوكيل دقيقة وكاملة ومفيدة بمرور الوقت. يمكن لوكيل مُقيّم أن يقيّم الاستجابات ويمنحها درجات وفقًا لمعايير محددة.

3. **إدارة التكاليف** — يؤثر استخدام التوكنات مباشرةً على التكلفة. تساعد استراتيجيات مثل تحسين المطالبات، اختيار النموذج، والتخزين المؤقت في إبقاء النفقات تحت السيطرة دون التضحية بالجودة.


## بناء وكيل قابل للمراقبة

نعرّف أدوات السفر ونغلف استدعاء الوكيل بقياس الزمن حتى نتمكن من مراقبة الكمون. في بيئة الإنتاج، ستندمج مع OpenTelemetry أو نظام تتبع خلفي مشابه.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## أنماط التقييم

نمط شائع في الإنتاج هو استخدام وكيل ثانٍ كـ **مقيّم**. يقوم المقيّم بتقييم استجابة الوكيل الأساسي بناءً على معايير محددة مسبقًا مثل الاكتمال، والدقة، والفائدة.

يتيح ذلك:
- بوابات جودة مؤتمتة قبل وصول الردود إلى المستخدمين
- اكتشاف التراجع عندما تتغير المطالبات أو النماذج
- المراقبة المستمرة لأداء الوكيل مع مرور الوقت


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتيجيات إدارة التكاليف

التحكم في التكاليف أمر حاسم لوكلاء الذكاء الاصطناعي في الإنتاج. فيما يلي استراتيجيات رئيسية:

| الاستراتيجية | الوصف |
|---|---|
| **تحسين المطالب** | اجعل تعليمات النظام موجزة. أزل السياق الزائد لتقليل عدد توكنات الإدخال. |
| **اختيار النموذج** | استخدم نماذج أصغر وأرخص (مثل GPT-4o-mini) للمهام البسيطة مثل التصنيف أو الاستخراج، واحتفظ بالنماذج الأكبر للمسائل التي تتطلب استدلالًا معقدًا. |
| **التخزين المؤقت** | قم بتخزين نتائج الأدوات والاستعلامات المتكررة لتجنب مكالمات API المكررة. |
| **حدود التوكن** | اضبط حدود `max_tokens` لمنع استجابات طويلة بشكل غير متوقع. |
| **التجميع** | اجمع استفسارات متعددة من المستخدم في مكالمة API واحدة عندما يكون ذلك ممكنًا. |

عمليًا، نهج متعدد الطبقات يعمل جيدًا: وجه الطلبات البسيطة إلى نموذج سريع ورخيص وصعّد الاستفسارات المعقدة فقط إلى نموذج أكثر قدرة.


## الملخص

في هذا الدرس تعلمت كيفية:

1. **إضافة قابلية الملاحظة** لتفاعلات الوكلاء عبر القياس الزمني والتسجيل، تمهيدًا للتتبع والمراقبة.
2. **تقييم ردود الوكيل** تلقائيًا باستخدام وكيل مقيم يُقيّم الاكتمال والدقة والفائدة.
3. **إدارة التكاليف** عبر تحسين الموجهات، اختيار النموذج، التخزين المؤقت، وميزانيات الرموز.

تساعد هذه الأنماط الإنتاجية على ضمان أن وكلاء الذكاء الاصطناعي لديك موثوقون، قابلون للقياس، وفعّالون من حيث التكلفة عند التوسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء المسؤولية**:
تم ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). وبينما نسعى إلى الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد. بالنسبة للمعلومات الحرجة، يُنصح بالاستعانة بترجمة بشرية احترافية. لا نتحمل أي مسؤولية عن أي سوء فهم أو تفسير خاطئ ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
